# Scraping Komentar X (Twitter)


In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from apify_client import ApifyClient

load_dotenv()

api_token = os.getenv("API_TOKEN")
x_cookies = os.getenv("X_COOKIES")

if not api_token:
    raise ValueError("API_TOKEN tidak ditemukan di .env")
if not x_cookies:
    raise ValueError("X_COOKIES tidak ditemukan di .env")

# Cookie boleh berupa header string mentah ("auth_token=...; ct0=...; twid=...")
if x_cookies.startswith("ENCRYPTED_VALUE:"):
    bagian = x_cookies.split("ENCRYPTED_VALUE:", 1)[1].split(":")
    if len(bagian) != 2 or any(len(b) % 4 for b in bagian):
        raise ValueError(
            "X_COOKIES (ENCRYPTED_VALUE) tidak utuh / terpotong. "
            "Salin ulang dari Apify, atau isi dengan cookie header mentah: "
            "auth_token=...; ct0=...; twid=..."
        )

client = ApifyClient(api_token)


## Konfigurasi


In [ ]:
ACTOR_ID = "muhammetakkurtt/twitter-x-comment-scraper"

# Satu URL tweet per baris. Boleh lebih dari satu, hasilnya digabung.
TWEET_URLS = [
    "https://x.com/triwul82/status/2116339327701795040?a=vK0u5mejQr_HKkmgeNg&s=20",
]

MAX_DEPTH = 3               # 0 = tanpa batas kedalaman balasan
ONLY_DIRECT_REPLIES = True  # hanya balasan langsung ke pemilik tweet
SORT_BY = "relevance"       # "relevance" | "newest" | "likes"


## Running


In [ ]:
all_items = []
failed = []

for tweet_url in TWEET_URLS:
    run_input = {
        "twitterUrl": tweet_url,
        "cookies": x_cookies,
        "maxDepth": MAX_DEPTH,
        "onlyDirectReplies": ONLY_DIRECT_REPLIES,
        "sortBy": SORT_BY,
    }

    try:
        run = client.actor(ACTOR_ID).call(run_input=run_input)
        items = list(client.dataset(run["defaultDatasetId"]).iterate_items())

        for item in items:
            item["sourceTweetUrl"] = tweet_url

        all_items.extend(items)
        print(f"{tweet_url}: {len(items)} komentar")
    except Exception as e:
        failed.append(tweet_url)
        print(f"{tweet_url}: GAGAL -> {e}")

df = pd.DataFrame(all_items)

# Buang duplikat kalau tweet yang sama di-scrape lebih dari sekali
if "tweet_id" in df.columns:
    df = df.drop_duplicates(subset=["tweet_id"], keep="first").reset_index(drop=True)

print(f"\nTotal komentar: {len(df)}")
if failed:
    print(f"Tweet gagal: {', '.join(failed)}")
if df.empty:
    print("Tidak ada komentar yang didapat. Cek cookie X (X_COOKIES) atau URL tweet-nya.")

df.head()


## Rapikan Kolom


In [ ]:
def ambil_user(u, key):
    return u.get(key) if isinstance(u, dict) else None


if "user" in df.columns:
    df["user_screen_name"] = df["user"].apply(lambda u: ambil_user(u, "screen_name"))
    df["user_name"] = df["user"].apply(lambda u: ambil_user(u, "name"))
    df["user_is_blue_verified"] = df["user"].apply(lambda u: ambil_user(u, "is_blue_verified"))

# Teks tweet sering mengandung baris baru; rapikan supaya satu baris = satu komentar
if "text" in df.columns:
    df["text"] = (
        df["text"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    )

KEEP_COLS = [
    "tweet_id",
    "tweet_url",
    "conversation_id",
    "in_reply_to_tweet_id",
    "text",
    "created_at",
    "lang",
    "favorite_count",
    "reply_count",
    "retweet_count",
    "quote_count",
    "views_count",
    "reply_to_username",
    "user_screen_name",
    "user_name",
    "user_is_blue_verified",
    "sourceTweetUrl",
]

df = df[[c for c in KEEP_COLS if c in df.columns]]

df.head()


In [ ]:
print(f"Jumlah data yang berhasil di scrape: {len(df)}")
df.isnull().sum()


## Simpan File


In [ ]:
os.makedirs("../results/x", exist_ok=True)

if df.empty:
    print("Data kosong, file tidak disimpan.")
else:
    df.to_csv("../results/x/x_comments.csv", index=False, encoding="utf-8-sig")
    df.to_excel("../results/x/x_comments.xlsx", index=False)

    print(f"Tersimpan: {len(df)} komentar -> x_comments.csv / .xlsx")
